<!-- CV-PATCHED -->

# ПЗ 2 — Нарезка видео на кадры

Скачиваем видео по ссылке, нарезаем на кадры с заданным шагом, сохраняем в Google Drive.

In [ ]:
!pip install -q opencv-python-headless yt-dlp tqdm 'numpy<2'


In [ ]:
# === Общий setup (Colab + локальный fallback) ===
import os, sys, subprocess, warnings, shutil
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=UserWarning)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = IN_COLAB and os.environ.get('CV_USE_DRIVE', '1') == '1'

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        BASE = '/content/drive/MyDrive/cv-frames'
    except Exception as e:
        print(f'[setup] drive недоступен ({e}) — пишем в /content/cv-frames')
        BASE = '/content/cv-frames'
elif IN_COLAB:
    BASE = '/content/cv-frames'
else:
    BASE = str((os.path.expanduser('~/cv-frames')))

BASE_DRIVE  = BASE
VIDEO_DIR   = f'{BASE}/видио'
FRAMES_DIR  = f'{BASE}/кадры'
RESULTS_DIR = f'{BASE}/результаты'
for d in (VIDEO_DIR, FRAMES_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
print(f'[setup] BASE={BASE}')


In [ ]:
# Пример: публичный Big Buck Bunny. Замените на свой URL при желании.
VIDEO_URL = 'https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_320x180.mp4'

# === Скачивание видео (yt-dlp работает и с прямыми URL, и с YouTube) ===
import os, subprocess
VIDEO_URL = globals().get('VIDEO_URL', 'https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_320x180.mp4')
video_path = f'{VIDEO_DIR}/video.mp4'

if os.path.exists(video_path) and os.path.getsize(video_path) > 0:
    print(f'[video] уже скачано: {video_path} ({os.path.getsize(video_path)/1e6:.1f} MB)')
else:
    try:
        r = subprocess.run(
            ['yt-dlp', '-f', 'mp4/best[ext=mp4]/best',
             '-o', video_path, '--no-playlist', VIDEO_URL],
            capture_output=True, text=True, timeout=180
        )
        if r.returncode != 0 or not os.path.exists(video_path):
            raise RuntimeError(r.stderr[-500:] if r.stderr else 'yt-dlp вернул код != 0')
        print(f'[video] скачано: {os.path.getsize(video_path)/1e6:.1f} MB')
    except Exception as e:
        print(f'[video] yt-dlp упал ({e}), пробуем прямой URL по умолчанию')
        import urllib.request
        urllib.request.urlretrieve('https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_320x180.mp4', video_path)
        print(f'[video] скачано fallback: {os.path.getsize(video_path)/1e6:.1f} MB')


In [4]:
# альтернатива — загрузить файл вручную
# from google.colab import files
# import shutil
# uploaded = files.upload()
# name = list(uploaded.keys())[0]
# video_path = f'{VIDEO_DIR}/{name}'
# shutil.move(name, video_path)

In [ ]:
import cv2
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise SystemExit('видео не открылось, проверьте файл/URL')
fps   = cap.get(cv2.CAP_PROP_FPS) or 25
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'fps: {fps:.1f}, кадров: {total}, длина: {total/fps:.1f} сек')


In [ ]:
from tqdm.notebook import tqdm

FRAME_STEP = 30

frame_idx = 0
saved = 0

for _ in tqdm(range(total), desc='нарезка'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1

cap.release()
print(f'сохранено {saved} кадров в {FRAMES_DIR}')

In [ ]:
from IPython.display import Image, display

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))[:3]
for f in frames:
    display(Image(filename=f'{FRAMES_DIR}/{f}', width=400))